# the Big Dipper isn't a thing
**standard deviation** · standardlydeviated.substack.com

Data: HYG Star Database v3.4 (https://github.com/astronexus/HYG-Database)  
Distances from Hipparcos parallax measurements.  
Proper motion from Hipparcos μα, μδ in mas/yr.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
# ---- star data ----
# Source: HYG v3.4, verified against SIMBAD June 2026
# Distances in light-years (converted from parallax)
# RA/Dec in decimal degrees (J2000)
# Proper motion in mas/yr

stars = pd.DataFrame({
    'name':    ['Megrez', 'Mizar',  'Merak',  'Alioth', 'Phecda', 'Dubhe',  'Alkaid'],
    'dist_ly': [58,       78,       79,        81,       84,       124,      210],
    'ra':      [183.8,    200.9,    165.5,     193.5,    178.5,    165.9,    206.9],
    'dec':     [57.0,     54.9,     56.4,      55.9,     53.7,     61.8,     49.3],
    'pm_ra':   [100,      121,       82,       112,       98,      -136,     -121],  # mas/yr
    'pm_dec':  [-21,      -24,      -35,       -26,      -20,       -37,     -101],  # mas/yr
    'group':   [True,     True,     True,      True,     True,     False,    False],
    'position':['bowl/handle join','handle middle','bowl btm right',
                'handle base','bowl btm left','bowl top right','handle tip'],
})

NOW = 2026
stars['light_departed'] = NOW - stars['dist_ly']
stars['color'] = stars['group'].map({True: '#8b85e8', False: '#e8855a'})
stars['group_label'] = stars['group'].map({True: 'UMa Moving Group', False: 'Unrelated'})

stars

In [ ]:
# ---- convert RA/Dec + distance to 3D Cartesian ----
ra_rad  = np.radians(stars['ra'])
dec_rad = np.radians(stars['dec'])
d       = stars['dist_ly']

stars['x3'] = d * np.cos(dec_rad) * np.cos(ra_rad)
stars['y3'] = d * np.cos(dec_rad) * np.sin(ra_rad)
stars['z3'] = d * np.sin(dec_rad)

print('3D positions computed.')
stars[['name','x3','y3','z3']].round(1)

In [ ]:
# ---- chart 1: 3D scatter — true positions ----
fig = go.Figure()

for _, row in stars.iterrows():
    fig.add_trace(go.Scatter3d(
        x=[row.x3], y=[row.y3], z=[row.z3],
        mode='markers+text',
        text=[row['name']],
        textposition='top center',
        marker=dict(size=8, color=row.color, opacity=0.9),
        name=row['name'],
        hovertemplate=f"<b>{row['name']}</b><br>{row.dist_ly} ly from Earth<br>Light left: {row.light_departed}<extra></extra>",
        showlegend=True,
    ))

# Add Earth marker
fig.add_trace(go.Scatter3d(
    x=[-170], y=[0], z=[30],
    mode='markers+text',
    text=['Earth'],
    textposition='top center',
    marker=dict(size=6, color='rgba(100,200,255,0.9)'),
    name='Earth',
    hovertemplate='Earth<extra></extra>',
))

fig.update_layout(
    title='Big Dipper stars — true 3D positions',
    scene=dict(
        bgcolor='#070714',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
    ),
    paper_bgcolor='#070714',
    font_color='rgba(255,255,255,0.8)',
    margin=dict(l=0, r=0, t=40, b=0),
    height=500,
)

fig.show()

In [ ]:
# ---- chart 2: distance bar chart ----
sorted_stars = stars.sort_values('dist_ly')

fig2 = go.Figure(go.Bar(
    x=sorted_stars['name'],
    y=sorted_stars['dist_ly'],
    marker_color=sorted_stars['color'],
    text=sorted_stars['dist_ly'].astype(str) + ' ly',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>%{y} light-years from Earth<br>Light departed: ' + 
                  sorted_stars['light_departed'].astype(str) + '<extra></extra>',
))

fig2.update_layout(
    title='Distance from Earth — Big Dipper stars',
    yaxis_title='light-years',
    plot_bgcolor='#070714',
    paper_bgcolor='#0a0a0f',
    font_color='rgba(255,255,255,0.8)',
    yaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
)

fig2.show()

In [ ]:
# ---- chart 3: pairwise 3D distances between stars ----
from itertools import combinations

pairs = []
for (i, a), (j, b) in combinations(stars.iterrows(), 2):
    d3 = np.sqrt((a.x3-b.x3)**2 + (a.y3-b.y3)**2 + (a.z3-b.z3)**2)
    pairs.append({
        'pair': f"{a['name']} → {b['name']}",
        'dist_3d': round(d3),
        'both_group': a.group and b.group,
    })

pairs_df = pd.DataFrame(pairs).sort_values('dist_3d')

fig3 = go.Figure(go.Bar(
    x=pairs_df['pair'],
    y=pairs_df['dist_3d'],
    marker_color=pairs_df['both_group'].map({True:'#8b85e8',False:'#e8855a'}),
    text=pairs_df['dist_3d'].astype(str) + ' ly',
    textposition='outside',
))

fig3.update_layout(
    title='Actual 3D distance between pairs of Big Dipper stars',
    yaxis_title='light-years (3D separation)',
    plot_bgcolor='#070714',
    paper_bgcolor='#0a0a0f',
    font_color='rgba(255,255,255,0.8)',
    xaxis_tickangle=-45,
    yaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
)

fig3.show()
pairs_df.head(10)

In [ ]:
# ---- stellar motion: project positions ±50,000 years ----
# Proper motion in mas/yr → angular displacement over time
# This is a simplified linear projection (ignores perspective acceleration)

years = np.linspace(-50000, 50000, 200)

# Scale factor: visual units per mas/yr per year (for illustration)
SCALE = 0.0004

# Base sky positions (simplified 2D: RA on x, Dec on y)
base_pos = {
    'Megrez': (0,   0),
    'Mizar':  (60,  40),
    'Merak':  (-80, -40),
    'Alioth': (30,  60),
    'Phecda': (-50, -60),
    'Dubhe':  (-90, 80),
    'Alkaid': (130, -50),
}

pm = dict(zip(stars['name'], zip(stars['pm_ra'], stars['pm_dec'])))

print('Proper motion data (mas/yr):')
for name, (pra, pdec) in pm.items():
    print(f'  {name:8s}: ra={pra:+.0f}, dec={pdec:+.0f}')

In [ ]:
# ---- travel time calculator ----
LY_TO_MILES = 5.879e12

speeds = {
    'Walking (3 mph)':               3,
    'Highway driving (70 mph)':      70,
    'Commercial flight (575 mph)':   575,
    'ISS orbital speed (17,500 mph)': 17500,
    'New Horizons probe (36,000 mph)': 36000,
    'Parker Solar Probe (430,000 mph)': 430000,
    'Speed of light (670,616,629 mph)': 670616629,
}

target_star = 'Alkaid'
dist_ly = stars.set_index('name').loc[target_star, 'dist_ly']
dist_miles = dist_ly * LY_TO_MILES

print(f'Travel times to {target_star} ({dist_ly} light-years):')
print(f'Distance: {dist_miles:.2e} miles\n')

for method, mph in speeds.items():
    hours = dist_miles / mph
    years = hours / 8760
    if years < 1000:
        fmt = f'{years:,.0f} years'
    elif years < 1e6:
        fmt = f'{years/1000:,.1f}K years'
    elif years < 1e9:
        fmt = f'{years/1e6:,.1f} million years'
    else:
        fmt = f'{years/1e9:,.2f} billion years'
    print(f'  {method:<45}: {fmt}')

In [ ]:
# ---- export static charts for Substack post ----
# Requires: pip install kaleido
# Uncomment to save:

# fig.write_html('big-dipper-3d.html')   # interactive, embed via GitHub Pages
# fig2.write_image('distances.png', scale=2)  # static, embed in Substack post
# fig3.write_image('pairwise-distances.png', scale=2)

print('Charts ready. Uncomment export lines above to save files.')